In [2]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
%matplotlib inline

In [9]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [10]:
len(words)

32033

In [11]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos) ; print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [ ]:
block_size = 3
def build_dataset(words):
    X,Y = [], []
    for w in words:
        context = [0] * block_size
        #print(w)
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            #print(''.join(itos[i] for i in context) , '--->', itos[ix])
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y
# -- SEPERATING DATASETS  --
import random
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [ ]:
# INITIALISATION BLOCK
n_emb = 10         
n_hidden = 200
C = torch.randn((27,2))
W1 = torch.randn((6,100))
B1 = torch.randn((100))
W2 = torch.randn((100,27))
B2 = torch.randn((27))
parameters = [C , W1, B1, W2, B2]
for p in parameters:
    p.requires_grad = True

In [7]:
# HOW TO DETERMINE A GOOD LEARNING RATE:
x = torch.linspace(0.001 , 1 , 1000)
lre = torch.linspace(-3, 0 , 1000) # WE USE THIS TO STEP EXPONENTIALLY INSTEAD OF LINEARLY
lrs = 10 ** lre
lri , lossi = [] , []

In [8]:
for i in range(1000):
    #---MINIBATCHING---
    ix = torch.randint(0, X.shape[0], (32,))
    #---FORWARD PASS---
    emb = C[X[ix]]
    h = torch.tanh(emb.view(-1,6) @ W1 + B1)
    logits = h @ W2 + B2
    loss = F.cross_entropy(logits,Y[ix])
    #print(loss.item())

    #---BACKWARD PASS---
    for p in parameters:
        p.grad = None
    loss.backward()

    #---UPDATING---
    lr = lrs[i]
    for p in parameters:
        p.data += -lr * p.grad

     #---TRACKING STATS--- 
    lri.append(lre[i])
    lossi.append(loss.item())